# Prosthetics Data Snippets

Small patterns you can copy into the participant workbook if you get stuck. Keep the parts you need and adapt the field names to the task.

In [ ]:
import pandas as pd

df = pd.read_csv('prosthetics_data.csv')
print(df.head())
print(df.info())
print(df.isna().sum())

In [ ]:
CURRENT_YEAR = 2026

def parse_age(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if text.isdigit():
        num = int(text)
        if num > 1900:          # birth year: convert to age
            return CURRENT_YEAR - num
        return num              # already an age
    # Text like "around 50" or "maybe 27" — pull the first number found
    digits = ''.join(character for character in text if character.isdigit())
    return int(digits) if digits else pd.NA

df['Age_Clean'] = df['Age_Input'].apply(parse_age)

In [ ]:
import re

def parse_date(raw):
    if pd.isna(raw):
        return pd.NA
    s = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', str(raw).strip())
    # Year-first formats (YYYY-MM-DD or YYYY/MM/DD)
    m = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if m:
        y, mo, d = m.groups()
        return f'{y}-{int(mo):02d}-{int(d):02d}'
    try:
        return pd.to_datetime(s, dayfirst=True).strftime('%Y-%m-%d')
    except Exception:
        return pd.NA

df['Implant_Date_Clean'] = df['Implant_Date'].apply(parse_date)
df['Amputation_Cause_Clean'] = df['Amputation_Cause'].astype(str).str.lower().str.strip()

In [ ]:
device_inventory = pd.DataFrame({
    'Device_Model_ID': ['A-100', 'B-200'],
    'Model_Name': ['LightStride', 'FlexStep'],
    'Cost_USD': [3400, 4800],
    'Manufacturer': ['Northbridge', 'Southfield'],
})
merged_df = pd.merge(df, device_inventory, on='Device_Model_ID', how='inner')

import hashlib

def hash_name(value):
    normalized = str(value).strip().lower()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

df['Full_Name_Hashed'] = df['Full_Name'].apply(hash_name)